# 00 Pipeline Overview

This notebook is the run map for the active hourly DA notebook pipeline.

How to use this folder:
- run notebooks `00` through `24` in order for the main pipeline
- notebooks `90+` are reference-only and sit outside the consecutive run order
- only `01_da_prices_cleaning_walkthrough.ipynb` and `03_endogenous_explicit_features.ipynb` are protected non-generator notebooks
- all other top-level notebooks in this folder are generator-managed and any ad hoc top-level `.ipynb` file can be archived on regeneration
- benchmark notebooks default to **reading the latest saved artifacts**
- the shared `find_latest_run(...)` helper prefers the latest complete run folder with `run_summary.json`
- use the execution hook at the end of an execution notebook only when you want to refresh that stage

Critical distinction:
- `01_da_prices_cleaning_walkthrough.ipynb` is where upstream raw missing-datapoint handling is explained
- `03_endogenous_explicit_features.ipynb` does **not** redo that cleaning step; it only creates a causal helper series on top of the cleaned target for leakage-safe feature engineering


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


In [ ]:
pipeline_rows = [
    {"step": "00", "notebook": "00_pipeline_overview.ipynb", "mode": "Guide", "ownership": "Generator-managed", "main_output": "Run order, dependencies, and latest artifact status"},
    {"step": "01", "notebook": "01_da_prices_cleaning_walkthrough.ipynb", "mode": "Preparation", "ownership": "Protected", "main_output": "Upstream DA cleaning walkthrough and missing-datapoint handling"},
    {"step": "02", "notebook": "02_methodology_and_objective_weeks.ipynb", "mode": "Preparation", "ownership": "Generator-managed", "main_output": "Frozen methodology snapshot and objective week selection"},
    {"step": "03", "notebook": "03_endogenous_explicit_features.ipynb", "mode": "Preparation", "ownership": "Protected", "main_output": "FS1 endogenous feature diagnostics and saved feature artifacts"},
    {"step": "04", "notebook": "04_fs0_naive_models.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "naive_benchmark"},
    {"step": "05", "notebook": "05_fs1_lear.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "lear_fs1_benchmark"},
    {"step": "06", "notebook": "06_fs1_xgboost.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "xgboost_fs1_benchmark"},
    {"step": "07", "notebook": "07_fs1_benchmark_and_comparison.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "fs1_model_comparison"},
    {"step": "08", "notebook": "08_fs2_lear.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "lear_fs2_benchmark"},
    {"step": "09", "notebook": "09_fs2_xgboost.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "xgboost_fs2_benchmark"},
    {"step": "10", "notebook": "10_fs2_prophet.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "prophet_benchmark"},
    {"step": "11", "notebook": "11_fs2_benchmark_and_shortlisting.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "model_comparison"},
    {"step": "12", "notebook": "12_fs2_lear_ablation.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "Model-specific FS2 staged block ablation for LEAR"},
    {"step": "13", "notebook": "13_fs2_xgboost_ablation.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "Model-specific FS2 staged block ablation for XGBoost"},
    {"step": "14", "notebook": "14_fs2_feature_value_results.ipynb", "mode": "Reporting", "ownership": "Generator-managed", "main_output": "FS2 staged ablation synthesis across LEAR and XGBoost"},
    {"step": "15", "notebook": "15_fs2_pruning_and_redesign_validation.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "Selective FS2 pruning and redesign validation against revised candidate parents"},
    {"step": "16", "notebook": "16_fs3_lear.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "FS3 LEAR parent benchmark on the combined all-exogenous family stack"},
    {"step": "17", "notebook": "17_fs3_xgboost.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "FS3 XGBoost parent benchmark on the combined all-exogenous family stack"},
    {"step": "18", "notebook": "18_fs3_benchmark_and_comparison.ipynb", "mode": "Reporting", "ownership": "Generator-managed", "main_output": "Cross-model comparison of the all-exogenous FS3 parent benchmarks against FS2"},
    {"step": "19", "notebook": "19_fs3_lear_ablation.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "Model-specific FS3 staged block ablation for LEAR"},
    {"step": "20", "notebook": "20_fs3_xgboost_ablation.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "Model-specific FS3 staged block ablation for XGBoost"},
    {"step": "21", "notebook": "21_fs3_feature_value_results.ipynb", "mode": "Reporting", "ownership": "Generator-managed", "main_output": "FS3 staged ablation synthesis across LEAR and XGBoost"},
    {"step": "22", "notebook": "22_fs3_pruning_and_redesign_validation.ipynb", "mode": "Execution", "ownership": "Generator-managed", "main_output": "Selective FS3 pruning and redesign validation against revised candidate parents"},
    {"step": "23", "notebook": "23_fs3_decision_relevant_forecast_evaluation.ipynb", "mode": "Reporting", "ownership": "Generator-managed", "main_output": "Decision-relevant DA candidate evaluation across FS2, promoted FS3, and pruned FS3 finalists"},
    {"step": "24", "notebook": "24_final_conclusion_and_best_model.ipynb", "mode": "Reporting", "ownership": "Generator-managed", "main_output": "Final thesis recommendation after the decision-relevant DA evaluation layer"},
]

reference_rows = [
    {"notebook": "90_active_stack_and_tuning_policy_reference.ipynb", "ownership": "Generator-managed reference", "purpose": "Background on active model-entry rules and tuning cadence"},
    {"notebook": "91_fs3_feature_family_plan_reference.ipynb", "ownership": "Generator-managed reference", "purpose": "Planned FS3 exogenous-family roadmap; not part of the active run order"},
    {"notebook": "92_fs4_huang_style_plan_reference.ipynb", "ownership": "Generator-managed reference", "purpose": "Planned FS4 advanced-feature roadmap; not part of the active run order"},
    {"notebook": "93_execution_readiness_checklist_reference.ipynb", "ownership": "Generator-managed reference", "purpose": "Optional structural repo check; not part of the main pipeline"},
    {"notebook": "94_methodology_update_summary_reference.ipynb", "ownership": "Generator-managed reference", "purpose": "Historical summary of the repo-standardization update"},
]

display(pd.DataFrame(pipeline_rows))
display(pd.DataFrame(reference_rows))


## Current artifact status


In [ ]:
rows = []
for run_label in ['case_week_selection', 'endogenous_explicit_features', 'naive_benchmark', 'lear_fs1_benchmark', 'xgboost_fs1_benchmark', 'fs1_model_comparison', 'lear_fs2_benchmark', 'xgboost_fs2_benchmark', 'prophet_benchmark', 'model_comparison', 'feature_family_ablation__lear_fs2_benchmark__stage_a_top_level__v1', 'feature_family_ablation__xgboost_fs2_benchmark__stage_a_top_level__v1', 'feature_family_ablation__lear_fs2_benchmark__layer1_mutually_exclusive__v1', 'feature_family_ablation__xgboost_fs2_benchmark__layer1_mutually_exclusive__v1', 'lear_fs2_pruned_candidate_benchmark', 'xgboost_fs2_pruned_candidate_benchmark', 'model_comparison_fs2_pruned_candidate', 'fs3_combo_promoted_confirm', 'lear_fs3_combo_promoted_benchmark', 'xgboost_fs3_combo_promoted_benchmark', 'lear_fs3_combo_pruned_candidate_benchmark', 'xgboost_fs3_combo_pruned_candidate_benchmark', 'feature_family_ablation__lear_fs1_benchmark', 'feature_family_ablation__xgboost_fs1_benchmark', 'feature_family_ablation__prophet_benchmark', 'visual_case_weeks']:
    run_dir = latest_run_or_none(run_label)
    rows.append(
        {
            "run_label": run_label,
            "latest_run": str(run_dir) if run_dir is not None else "not yet run",
        }
    )

display(pd.DataFrame(rows))
